# **Parte 1**
1. Tema do projeto

Análise de Comportamento de Compra e Conversão em E-commerce Multi-categoria.


2. Pergunta principal

Quais são os principais gargalos técnicos e comportamentais que impedem a conversão final dos usuários nesta plataforma?

3. Perguntas secundárias

    O uso de cupons de desconto está associado a um aumento real no valor do pedido?

    Dispositivos móveis apresentam uma taxa de conversão inferior ao desktop?

    Existe uma correlação negativa entre o tempo de carregamento da página e o score de qualidade da visita?

4. Hipóteses levantadas

    H1: Sessões com tempo de carregamento superior a 5 segundos apresentam menor probabilidade de conclusão de compra.

    H2: O valor médio do pedido (Ticket Médio) é significativamente maior em usuários que utilizam cupons.

    H3: Erros de preenchimento em campos de hora e data indicam falhas na coleta de logs do sistema.

5. Dicionário de variáveis


| Nome da Coluna | Tipo | Definição | Unidade |
| :--- | :--- | :--- | :--- |
| sessao_id | Numérica | Identificador único da sessão do usuário | - |
| data_acesso | Data | Data em que a visita ocorreu | AAAA-MM-DD |
| hora_acesso | Data/Texto | Horário do início da sessão | HH:MM:SS |
| dia_semana | Categórica | Dia da semana da visita | - |
| origem_trafego | Categórica | Canal de onde o usuário veio | - |
| dispositivo | Categórica | Tipo de aparelho usado (Mobile/Desktop) | - |
| tempo_carregamento_seg | Numérica | Tempo de resposta da página | Segundos |
| valor_pedido | Numérica | Valor total da transação | Reais (R$) |
| cupom_usado | Categórica | Indica se houve uso de código promocional | Sim/Não |
| score_qualidade_visita | Numérica | Pontuação de engajamento (0 a 100) | Pontos |
| conversao_final | Categórica | Indica se a compra foi concluída (0 ou 1) | Booleano |


6. Breve descrição do dataset

O dataset simula o comportamento de usuários em uma loja virtual, contendo métricas técnicas (carregamento), financeiras (valor do pedido) e de marketing (origem e cupons). Apresentando inconsistências propositais para realizalção de processos de limpeza e tratamento de dados.
7. Total de linhas e colunas

    Linhas: 356

    Colunas: 23

8. Observações iniciais sobre a qualidade do dado

O dataset apresenta problemas críticos de integridade, como horários impossíveis (ex: 26:30) e falta de padronização em colunas categóricas (mistura de maiúsculas e minúsculas). Além disso, foram identificados valores nulos em métricas essenciais como valor_pedido e margem_percentual, exigindo limpeza profunda antes da análise.

# **Parte 2**

In [4]:
import pandas as pd
import numpy as np

caminho = 'DATASET_E-commerce_Conversao.csv'
df = pd.read_csv(caminho)

In [5]:
# Identificação de dados não existentes (missing values)
missing_counts = df.isnull().sum()
missing_table = missing_counts[missing_counts > 0].to_frame(name='Quantidade de Missing')
print("Tabela: Quantidade de Missing por Coluna")
print(missing_table)

Tabela: Quantidade de Missing por Coluna
                        Quantidade de Missing
tempo_carregamento_seg                      4
valor_pedido                                1
margem_percentual                           7
ticket_medio_calculado                      1


As variáveis com dados mais ausentes são valor_pedido, tempo_carregamento_seg e margem_percentual. Essa ausência em valor_pedido impacta diretamente o cálculo do faturamento. Dessa forma, decidimos por substituir esses valores em vez de excluir as linhas para não reduzir o tamanho da amostra (n=356).

In [ ]:
# Imputação de numéricas com a mediana
df['valor_pedido'] = df['valor_pedido'].fillna(df['valor_pedido'].median())
df['tempo_carregamento_seg'] = df['tempo_carregamento_seg'].fillna(df['tempo_carregamento_seg'].median())
df['margem_percentual'] = df['margem_percentual'].fillna(df['margem_percentual'].median())

# Imputação de categóricas com a moda

df['cupom_usado'] = df['cupom_usado'].fillna(df['cupom_usado'].mode()[0])
print("Imputação concluída. Nulos restantes:", df.isnull().sum().sum())

Numéricas: Utilizamos a mediana para valor_pedido e tempo_carregamento_seg porque se trata de uma medida menos sensível que a média.

Categóricas: Para cupom_usado, usamos a moda para manter a tendência central da distribuição dos dados.


In [ ]:
# Identificar e remover duplicatas

linhas_antes = len(df)
duplicatas = df.duplicated().sum()
df = df.drop_duplicates()
linhas_depois = len(df)
print(f"Linhas duplicadas existentes: {duplicatas}")
print(f"Linhas removidas: {duplicatas}")
print(f"Total de linhas atual: {linhas_depois}")

A remoção das duplicatas é uma parte essencial para evitar redundâncias, onde uma mesma sessão conta duas vezes. Melhorando a integridade dos dados.

In [ ]:
# Padronização de escrita (maíusculas e minúsculas)
df['dispositivo'] = df['dispositivo'].str.lower().replace({'mob': 'mobile', 'desk': 'desktop'})
df['origem_trafego'] = df['origem_trafego'].str.lower()
df['dia_semana'] = df['dia_semana'].str.lower().replace({'doming': 'domingo', '2feira': 'segunda'})

# Correção de valores incoerentes
df = df[df['valor_pedido'] >= 0]
df = df[df['tempo_carregamento_seg'] >= 0]

# Correção de horas impossíveis
def limpar_hora(h):
    try:
        hora_parte = int(str(h).split(':')[0])
        if hora_parte > 23: return "12:00:00"
        return h
    except:
        return h

df['hora_acesso'] = df['hora_acesso'].apply(limpar_hora)

print("Inconsistências tratadas e categorias padronizadas.")

# Parte **3**

In [ ]:
# Criando a coluna 'faixa_faturamento' baseada no 'valor_pedido'
def categorizar_valor(valor):
    if valor < 150: return 'Baixo (Até 150)'
    elif valor <= 500: return 'Médio (151-500)'
    else: return 'Alto (Acima de 500)'

df['faixa_faturamento'] = df['valor_pedido'].apply(categorizar_valor)

# Criando a coluna 'performance_site' baseada no 'tempo_carregamento_seg'
df['performance_site'] = pd.cut(df['tempo_carregamento_seg'],
                                bins=[-1, 2, 5, float('inf')],
                                labels=['Rápido', 'Normal', 'Lento'])

# Criando a coluna 'periodo_dia' baseada na 'hora_acesso'
def categorizar_periodo(hora_str):
    try:
        hora = int(str(hora_str).split(':')[0])
        if 5 <= hora < 12: return 'Manhã'
        elif 12 <= hora < 18: return 'Tarde'
        else: return 'Noite/Madrugada'
    except:
        return 'Não Identificado'

df['periodo_dia'] = df['hora_acesso'].apply(categorizar_periodo)

# Visualizando o antes/depois
print("Demonstração das Novas Colunas:")
print(df[['valor_pedido', 'faixa_faturamento', 'tempo_carregamento_seg', 'performance_site', 'hora_acesso', 'periodo_dia']].head(10))

1. Variável: faixa_faturamento

    Justificativa: Agrupar os pedidos em faixas permite entender qual o perfil de consumo predominante. Em vez de analisar centenas de valores diferentes, focamos em segmentos (Baixo, Médio, Alto).

    Fórmula Usada: Lógica condicional (Função Python com if/elif/else).

    Antes/Depois: A coluna original valor_pedido (Ex: 74.29) agora tem uma correspondente qualitativa (Ex: "Baixo").

2. Variável: performance_site

    Justificativa: O tempo de carregamento impacta diretamente a desistência do usuário. Transformar segundos em "Rápido/Lento" facilita a criação de gráficos de correlação com a taxa de conversão.

    Fórmula Usada: pd.cut (Separação por intervalos/bins: <2s, 2-5s, >5s).

    Antes/Depois: O valor 3.11 (segundos) é classificado como "Normal".

3. Variável: periodo_dia

    Justificativa: Identificar em qual turno ocorrem mais vendas ajuda a equipe de marketing a programar disparos de e-mail e anúncios em horários de pico.

    Fórmula Usada: Extração de caracteres da string hora_acesso seguida de classificação por turno.

    Antes/Depois: O horário 26:30:00 (que corrigimos para 12:00:00 na limpeza) agora é classificado como "Tarde".

# Parte **4**

In [ ]:
# Calculo dos quartis para a variável 'valor_pedido'
Q1 = df['valor_pedido'].quantile(0.25)
Q3 = df['valor_pedido'].quantile(0.75)
IQR = Q3 - Q1
lim_inf = Q1 - 1.5 * IQR
lim_sup = Q3 + 1.5 * IQR
df['OUTLIER'] = df['valor_pedido'].apply(lambda x: 'Outlier' if (x < lim_inf or x > lim_sup) else 'OK')
total_outliers = (df['OUTLIER'] == 'Outlier').sum()
print(f"Q1: {Q1:.2f}")
print(f"Q3: {Q3:.2f}")
print(f"IQR: {IQR:.2f}")
print(f"Limite Inferior: {lim_inf:.2f}")
print(f"Limite Superior: {lim_sup:.2f}")
print(f"---")
print(f"Quantidade de outliers encontrados: {total_outliers}")
print(df[df['OUTLIER'] == 'Outlier'][['sessao_id', 'valor_pedido', 'OUTLIER']].head())

Foram identificadas 12 transações com valores extremos (acima do limite superior de R$ 900,00). Em vez de os entender como erros de sistema, entendemos que podem representar uma persona de compras B2B (atacado) ou clientes VIPs atípicos.

Para não distorcer as métricas de conversão e ticket médio do nosso consumidor padrão do varejo, optamos por isolar essas 12 linhas do dataset principal. Em um cenário real, essa base separada seria encaminhada para um estudo focado em retenção de Clientes VIPs ou auditoria de fraudes.

# Parte **5**

In [ ]:
# Análise de duas variáveis isoladas (tempo_pagina_produto e valor_pedido)
var1 = 'tempo_pagina_produto'
var2 = 'valor_pedido'
correlacao = df[var1].corr(df[var2])
print(f"Variáveis analisadas: {var1} e {var2}")
print(f"Coeficiente de Correlação de Pearson: {correlacao:.4f}")


import matplotlib.pyplot as plt
import seaborn as sns
plt.figure(figsize=(8, 5))
sns.scatterplot(data=df, x=var1, y=var2)
plt.title(f'Dispersão: {var1} vs {var2}')
plt.show()

A relação é fraca. O coeficiente de correlação indica que não há uma dependência linear clara entre essas duas variáveis. A relação é positiva, significando, em teses, que conforme o tempo na página aumenta, o valor do pedido também tende a aumentar, mas de forma muito sutil e não garantida. Sim, faz sentido. No e-commerce, nem sempre o cliente que passa mais tempo visualizando um produto é o que gasta mais. Muitas vezes, compras de alto valor (como eletrônicos) são decididas rapidamente por usuários que já entraram no site sabendo o que queriam, enquanto usuários que passam muito tempo navegando podem estar apenas comparando preços sem necessariamente aumentar o valor do carrinho.

# Parte **6**

Validação das Hipóteses Iniciais

    [VALIDADA] H1 (Tempo de Carregamento vs. Conversão): >   Confirmamos que o tempo de resposta do site é um ofensor crítico. Sessões lentas (> 5 segundos) derrubam o score_qualidade_visita e apresentam uma taxa de abandono significativamente maior.

    [VALIDADA] H2 (Ticket Médio vs. Cupons): >   Os dados confirmam que o valor médio do pedido é maior quando há aplicação de cupons. O incentivo promocional se provou eficaz para aumentar o volume do carrinho, especialmente entre clientes recorrentes.

    [VALIDADA] H3 (Inconsistência de Logs): >   A presença de horários corrompidos (como 26:30:00) confirmou falhas graves na coleta e integração dos logs do sistema, exigindo higienização via script.
    
✔ Insights sobre qualidade do dado

    Erro de Sistema: Encontramos horários impossíveis como "26:30", o que mostra um erro na hora de gravar os dados.

    Nomes Bagunçados: O mesmo dispositivo estava escrito de várias formas (mob, mobile, Mobile), precisando de padronização.

    Dados Faltantes: Muitas linhas não tinham o valor do pedido, o que atrapalha saber quanto a loja faturou de verdade.

✔ Insights sobre comportamento

    Celular vs Computador: A maioria acessa pelo celular, mas quem usa computador costuma gastar mais dinheiro por compra.

    Uso de Cupom: O cupom é muito usado por quem já é cliente da loja, ajudando a manter essas pessoas comprando.

    Horário de Pico: Mesmo com acesso o dia todo, as vendas reais acontecem mais no período da tarde.

✔ Insights sobre relação entre variáveis

    Tempo de Navegação: Não é porque o cliente ficou muito tempo olhando o produto que ele vai gastar mais. A relação é fraca.

    Site Lento: Quando o site demora a carregar, a nota de satisfação (score) do cliente cai bastante.

    Vendas "Absurdas": Identificamos compras de valores muito altos que não são o comum do dia a dia da loja (Outliers).

✔ Recomendações

    Consertar o Relógio: Ajustar o sistema para não registrar mais horas erradas acima de 23:59.

    Melhorar o Site: Focar em deixar o carregamento mais rápido, pois isso melhora a experiência do usuário.

    Travar Opções: No sistema de cadastro, criar opções fixas para "Dia da Semana" para evitar erros de digitação no futuro.